**Note on replication**

The data this notebook processes were created in from the same commit this notebook was first added to Git

In [29]:
@file:DependsOn("../build/libs/processing.jar")
import profilelib.*

In [30]:
typealias Data =  Map<Map<String, String>, Map<String, Int>>

In [31]:
import jdk.jfr.consumer.RecordedFrame

fun load_data(dataPath: String): Data =
    walkPath(dataPath)
        .filter { it.endsWith(".jfr") }
        .map { path ->
            val params = path
                .removeSuffix(".jfr")
                .substringAfterLast("/")
                .split(",")
                .map { it.split("=").let { (k, v) -> k to v} }
                .toMap()
            val freq = mutableMapOf<String, Int>()
            println("'" + path + "'")
            lazy_samples(path).toFreq(freq, when (params["platform"]) {
                "jvm" -> ::jvm_get_name
                "native" -> fun(frame: RecordedFrame): String = frame.method.name
                    .let { if (it.startsWith("kfun:"))
                        it.removePrefix("kfun:").replace("#", ".").substringBefore("(")
                    else
                        "weirdo:" + it
                    }
                else -> throw NotImplementedError("Unknown platform: ${params["platform"]}")
            })
            params to freq.toMap()
        }
        .toMap()

In [32]:
val data = load_data("../../jfrStorage/serialization-twitterBM/warmup")

'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=43,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=46,warmup=2.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=52,warmup=10.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=1,warmup=0.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=35,warmup=10.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=38,warmup=10.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=native,cycles=100000,sampleInterval=100,iteration=10.jfr'
'../../jfrStorage/serialization-twitterBM/warmup/platform=jvm,cycles=100000,sampleInterval=100,iteration=61,warmup=2.jfr'
'../../jfrStorage/serializat

In [33]:
for (warmup in listOf(0, 1, 2, 3, 5, 10)) {
    print(warmup)
    print("\t")
    for (selector in listOf(
        fun(filtered_data: Data) = filtered_data.map { it.value["org.jetbrains.stdlibprofiling.TwitterBenchmark.encodeDecode"]!!.toDouble()},
        fun(filtered_data: Data) = filtered_data.map { it.value["java.lang.String.substring"]!!.toDouble()},
        fun(filtered_data: Data) = filtered_data.map {
            it.value["java.lang.String.substring"]!!.toDouble() /
                    it.key["iteration"]!!
                        .let { iteration ->
                            data.filter { it.key["platform"]!! == "native" && it.key["iteration"]!! == iteration }
                                .toList().single().second
                        }
                        .let { it["weirdo:Kotlin_String_subSequence"]!!.toDouble() }
        }
        )) {
            data.filter { it.key["platform"]!! == "jvm" && it.key["warmup"]!! == warmup.toString() }
                .let{ selector(it).toList() }
                .let {
                    val mean = it.sum().toDouble() / it.size
                    val variance = it.sumOf { (it - mean).let { it * it } } / it.size
                    val stddev = Math.sqrt(variance)
                    val cv = stddev / mean
                    "%.3f (%.3f)   ".format(mean, cv)
                }
                .let { print(it) }
        }
     println()
}

0	4525.025 (0.022)   209.750 (0.075)   1.021 (0.100)   
1	4466.988 (0.023)   206.538 (0.071)   1.007 (0.113)   
2	4464.125 (0.024)   204.950 (0.083)   0.999 (0.116)   
3	4464.375 (0.027)   202.263 (0.080)   0.986 (0.110)   
5	4461.050 (0.022)   205.900 (0.069)   1.003 (0.097)   
10	4470.438 (0.023)   204.550 (0.073)   0.997 (0.106)   


There doesn't seem to be a clear sign that the number of warmup iterations influence the results significantly. It seems
that one warmup iteration (with 100 000 cycles) is enough.

So, the warmup doesn't seem to explain why `new_base` measurement produced such a different results. This measurement
should be practically identical to the first made (`sampleInterval_and_cycles_variance`) but still shows different
results. Further analysis is required.